# Hawa: three years of PM2.5 in ten Indian cities

This notebook walks through the analysis step by step. The reusable functions live in `src/hawa/analysis.py` and are unit-tested in `tests/`.

**Data:** hourly PM2.5 from the Open-Meteo Air Quality API for 2023 to 2025. The values come from the CAMS global atmospheric model on a grid of roughly 40 km, not from ground monitors. They describe regional background air well and street-level pollution poorly.

In [1]:
import pandas as pd
from hawa import analysis
from hawa.fetch import RAW_PATH

pd.set_option('display.precision', 1)
hourly = analysis.load(RAW_PATH)
hourly.head()

,city,time,pm2_5,pm10
0,Delhi,2023-01-01 00:00:00,87.5,136.4
1,Delhi,2023-01-01 01:00:00,87.5,135.0
2,Delhi,2023-01-01 02:00:00,87.8,134.1
3,Delhi,2023-01-01 03:00:00,90.8,136.9
4,Delhi,2023-01-01 04:00:00,94.1,140.6


## 1. Check the data before trusting it

Three checks: the shape is what we expect (10 cities x 3 years x 24 hours), nothing is missing, and no value is negative.

In [2]:
expected = 10 * (365 + 366 + 365) * 24
print('rows:', len(hourly), '| expected:', expected)
print('missing values:', int(hourly.isna().sum().sum()))
print('negative readings:', int((hourly[['pm2_5', 'pm10']] < 0).sum().sum()))
hourly.groupby('city')['pm2_5'].describe()

rows: 263040 | expected: 263040
missing values: 0
negative readings: 0


,count,mean,std,min,25%,50%,75%,max
city,,,,,,,,
Ahmedabad,26304.0,29.1,18.4,2.3,15.7,23.2,37.8,136.4
Bengaluru,26304.0,20.9,14.5,0.6,8.7,18.6,29.7,92.2
Bhilai,26304.0,50.9,31.2,3.3,27.5,43.3,67.3,229.1
Chennai,26304.0,22.5,12.5,1.4,13.3,20.0,29.3,133.6
Delhi,26304.0,70.8,43.1,4.6,40.9,59.6,88.9,369.7
Hyderabad,26304.0,27.6,17.6,1.0,13.2,24.6,38.5,105.2
Kolkata,26304.0,56.6,47.0,2.5,21.3,42.1,77.8,363.2
Lucknow,26304.0,56.4,37.8,3.5,30.9,44.7,70.3,325.0
Mumbai,26304.0,39.2,30.0,3.7,19.6,28.2,52.0,581.1


The largest single reading is worth a look before deciding whether to keep it.

In [3]:
hourly.loc[[hourly['pm2_5'].idxmax()]]

,city,time,pm2_5,pm10
159543,Mumbai,2023-03-13 15:00:00,581.1,832.6


Mumbai, 13 March 2023. PM10 is extreme in the same hour, which points to a dust event more than a data error, so the row stays. One hour among 26,304 barely moves a three-year mean either way.

## 2. From hours to days

Air quality limits are written for 24-hour averages, so the unit of analysis is the day. A day needs at least 18 hourly readings to count.

In [4]:
daily = analysis.daily_means(hourly)
print(len(daily), 'city-days')
daily.head()

10960 city-days


,city,date,pm2_5
0,Ahmedabad,2023-01-01,53.1
1,Ahmedabad,2023-01-02,62.9
2,Ahmedabad,2023-01-03,51.2
3,Ahmedabad,2023-01-04,41.9
4,Ahmedabad,2023-01-05,45.8


## 3. How bad, and how often?

India's annual PM2.5 limit is 40 micrograms per cubic metre and the WHO guideline is 5. The daily limits are 60 and 15.

In [5]:
summary = analysis.city_summary(daily)
summary

,mean,pct_days_over_naaqs,pct_days_over_who,times_who_annual,times_naaqs_annual
city,,,,,
Delhi,70.8,51.2,100.0,14.2,1.8
Patna,59.5,38.0,98.4,11.9,1.5
Kolkata,56.6,38.3,88.9,11.3,1.4
Lucknow,56.4,32.0,99.4,11.3,1.4
Bhilai,50.9,31.1,95.8,10.2,1.3
Mumbai,39.2,19.0,94.8,7.8,1.0
Ahmedabad,29.1,5.1,84.5,5.8,0.7
Hyderabad,27.6,2.5,72.2,5.5,0.7
Chennai,22.5,0.4,71.4,4.5,0.6


Five cities exceed India's own annual limit. **Every** city is more than four times over the WHO guideline, including Bengaluru, the cleanest. Delhi is over the Indian daily limit on about half of all days.

Bhilai, a steel town of about a million people, ranks fifth. Its average is closer to Lucknow's than to Mumbai's.

## 4. Season

Mean PM2.5 by month, then the ratio of winter (Nov to Jan) to monsoon (Jun to Aug).

In [6]:
analysis.monthly_profile(daily).loc[summary.index].round(0).astype(int)

month,1,2,3,4,5,6,7,8,9,10,11,12
city,,,,,,,,,,,,
Delhi,121,70,65,52,64,59,48,49,44,63,104,109
Patna,114,74,54,38,46,43,29,32,33,53,95,104
Kolkata,123,78,57,44,29,33,17,26,23,53,95,103
Lucknow,107,61,49,36,45,42,30,31,34,51,91,99
Bhilai,84,58,55,46,42,34,23,27,32,54,82,76
Mumbai,68,59,52,34,24,20,21,18,19,42,58,56
Ahmedabad,44,31,25,21,20,18,19,17,21,39,51,42
Hyderabad,41,32,30,31,27,17,8,14,17,31,43,41
Chennai,33,27,22,18,23,24,14,19,18,21,23,28


In [7]:
analysis.seasonal_ratio(daily).to_frame()

,winter_to_monsoon
city,
Kolkata,4.2
Hyderabad,3.1
Bengaluru,3.1
Mumbai,3.1
Patna,3.0
Bhilai,2.9
Lucknow,2.9
Ahmedabad,2.5
Delhi,2.1


Every city peaks between November and January and bottoms out in the monsoon. Kolkata has the widest swing: its winter air carries more than four times the PM2.5 of its monsoon air. Chennai has the narrowest, likely because its main rains arrive with the north-east monsoon in October to December, in the middle of everyone else's dirty season.

The swing matters for policy: an annual average hides the fact that most of the exposure is packed into three months.

## 5. Longest unbroken bad spell

In [8]:
analysis.longest_streak(daily).to_frame()

,longest_streak_days
city,
Patna,61
Kolkata,60
Delhi,45
Lucknow,41
Bhilai,32
Mumbai,22
Ahmedabad,8
Hyderabad,6
Chennai,2


Patna and Kolkata each had about two months without a single day under the Indian limit. Bhilai's longest spell was 32 days.

## 6. Time of day

Each city's hours are divided by that city's own mean, so shapes can be compared between clean and dirty cities.

In [9]:
profile = analysis.diurnal_profile(hourly)
pd.DataFrame({'peak hour': profile.idxmax(), 'cleanest hour': profile.idxmin(), 'peak / trough': profile.max() / profile.min()}).sort_values('peak / trough', ascending=False)

,peak hour,cleanest hour,peak / trough
city,,,
Bhilai,22,15,1.8
Delhi,22,15,1.8
Lucknow,22,15,1.7
Kolkata,21,14,1.7
Patna,22,15,1.6
Bengaluru,20,14,1.6
Ahmedabad,21,16,1.6
Hyderabad,21,15,1.6
Mumbai,13,4,1.5


Eight cities share one shape: cleanest in mid-afternoon, dirtiest between 20:00 and 22:00. That matches a well-known mechanism. Daytime heating deepens the mixing layer and dilutes pollution; after sunset the layer collapses and emissions are trapped near the ground.

Two cities break the pattern. Chennai is nearly flat across the day (peak only 12% above trough). Mumbai is inverted, peaking at 13:00. I do not have an explanation for Mumbai that I can support with this dataset. A coastal sea-breeze effect is plausible, and so is an artefact of how a 40 km model cell that is half ocean behaves. It needs ground-monitor data to settle.

## 7. Year on year

In [10]:
yearly = analysis.yearly_means(daily)
yearly['change 2023 to 2025, %'] = (yearly[2025] / yearly[2023] - 1) * 100
yearly.sort_values('change 2023 to 2025, %')

year,2023,2024,2025,"change 2023 to 2025, %"
city,,,,
Mumbai,46.4,35.2,36.0,-22.5
Chennai,25.7,19.9,21.8,-15.4
Ahmedabad,30.2,28.6,28.3,-6.3
Bhilai,52.6,49.5,50.6,-3.8
Hyderabad,28.5,26.0,28.2,-1.0
Bengaluru,21.4,19.5,21.8,2.2
Kolkata,56.6,52.6,60.7,7.2
Patna,58.2,57.4,62.9,7.9
Lucknow,54.6,54.1,60.4,10.6


Delhi's mean rose about 21% and Mumbai's fell about 22%. Three years is too short to call either a trend: one unusually wet or still winter moves an annual mean this much.

## 8. Diwali: a result that did not appear

I expected a clear spike. The test compares Diwali day and the day after with the week before.

In [11]:
effect = analysis.diwali_effect(daily)
effect.pivot(index='city', columns='year', values='ratio').loc[summary.index].round(2)

year,2023,2024,2025
city,,,
Delhi,0.5,0.8,1.4
Patna,0.8,1.5,1.6
Kolkata,0.7,1.2,0.7
Lucknow,0.6,1.2,1.2
Bhilai,1.2,2.0,0.7
Mumbai,0.9,1.1,1.5
Ahmedabad,0.5,0.8,1.4
Hyderabad,2.3,0.8,0.3
Chennai,4.0,0.5,0.6


In [12]:
print('city-years with dirtier air on Diwali:', int((effect['ratio'] > 1).sum()), 'of', len(effect))
print('median ratio:', round(effect['ratio'].median(), 2))

city-years with dirtier air on Diwali: 14 of 30
median ratio: 0.93


There is no consistent spike. In Delhi the model shows *cleaner* air on Diwali 2023 than the week before.

The reason is the data source, not the fireworks. CAMS computes pollution from emission inventories and weather. Fireworks are a short local burst that is not in those inventories, so the model cannot see them. What the ratio actually picks up is weather: Delhi had rain just before Diwali 2023.

**Conclusion:** this dataset cannot answer the Diwali question. Answering it needs hourly ground-monitor data from CPCB stations. I kept the null result in because knowing what a dataset cannot tell you is part of the analysis.

## Summary

1. Five of ten cities exceed India's annual PM2.5 limit; all ten exceed the WHO guideline by more than four times.
2. Pollution is seasonal everywhere. Winter air carries 1.5 to 4.2 times the PM2.5 of monsoon air.
3. Patna and Kolkata each went about two months without one day under the limit.
4. Nights are dirtier than afternoons in eight cities. Chennai is nearly flat and Mumbai is inverted, which this dataset cannot explain.
5. Model data cannot detect Diwali fireworks. That question needs ground monitors.